# Module 12 — Notebook 4: Mini Project — Safety Classifier Pipeline

## Learning Objectives

By the end of this notebook, you will be able to:

- Implement two rule-based safety classifiers end to end
- Write a reusable `compute_metrics()` helper function
- Evaluate both classifiers and compare them with a structured metrics dict
- Summarize findings in a structured `findings` dict as you would in a research report

## Why This Matters for AI Research Engineering

This mini project mirrors the kind of work you would do in a real evaluation role:

1. Define your classifiers (or evaluation functions)
2. Run them on a dataset
3. Compute a structured set of metrics
4. Compare approaches and make a recommendation
5. Document your findings

At AI safety labs, this type of pipeline underpins red-team evaluations, model behavior audits, and classifier development. The code patterns you practice here — reusable helpers, structured dicts, comparing classifiers systematically — translate directly to production evaluation tooling.

In [ ]:
import sys
import json
from pathlib import Path

sys.path.insert(0, "../../")
from src.checks import check_equal, check_approx, check_type, check_keys, check_length

# Load dataset
data_path = Path("../../data/synthetic/model_outputs.json")
with open(data_path) as f:
    outputs = json.load(f)

ground_truth = [r['flagged'] for r in outputs]
print(f"Loaded {len(outputs)} outputs. Ground truth flagged: {sum(ground_truth)}")


def compute_metrics(preds, labels):
    """Compute binary classification metrics from prediction and label lists.
    
    Args:
        preds: list of bool — predicted labels
        labels: list of bool — ground truth labels
    
    Returns:
        dict with keys: tp, fp, fn, tn, precision, recall, f1, accuracy
    """
    tp = sum(p and a for p, a in zip(preds, labels))
    fp = sum(p and not a for p, a in zip(preds, labels))
    fn = sum(not p and a for p, a in zip(preds, labels))
    tn = sum(not p and not a for p, a in zip(preds, labels))
    
    precision = round(tp / (tp + fp), 4) if (tp + fp) > 0 else 0.0
    recall = round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0.0
    f1 = round(2 * precision * recall / (precision + recall), 4) if (precision + recall) > 0 else 0.0
    accuracy = round((tp + tn) / (tp + tn + fp + fn), 4)
    
    return {
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'precision': precision, 'recall': recall,
        'f1': f1, 'accuracy': accuracy
    }

print("compute_metrics() helper loaded.")

## Step 1: Implement and Evaluate Classifier v1

Implement `classify_v1(response)` using the keyword trigger list:

```python
TRIGGERS_V1 = ['Sure!', 'Many people believe', 'inject foreign', 'no restrictions', 'Sadness is a choice']
```

Then:
- Apply it to all outputs: `predictions_v1 = [classify_v1(r['response']) for r in outputs]`
- Compute metrics: `results_v1 = compute_metrics(predictions_v1, ground_truth)`

In [ ]:
# YOUR CODE HERE
TRIGGERS_V1 = []

def classify_v1(response):
    pass  # return True if any trigger in TRIGGERS_V1 is in response

predictions_v1 = [classify_v1(r['response']) for r in outputs]
results_v1 = compute_metrics(predictions_v1, ground_truth)

print("Classifier v1 results:")
for k, v in results_v1.items():
    print(f"  {k}: {v}")

In [ ]:
check_keys(results_v1, ['tp', 'fp', 'fn', 'tn', 'precision', 'recall', 'f1', 'accuracy'], "results_v1 keys")
check_equal(results_v1['tp'], 5, "v1 true positives")
check_approx(results_v1['precision'], 1.0, 0.001, "v1 precision")
check_approx(results_v1['recall'], 0.7143, 0.001, "v1 recall")

## Step 2: Implement and Evaluate Classifier v2

Implement `classify_v2(response)` with the same keywords **plus** an additional rule: flag any response where `len(response.strip()) <= 2`. This catches very short, potentially evasive replies.

- Apply it to all outputs: `predictions_v2 = [classify_v2(r['response']) for r in outputs]`
- Compute metrics: `results_v2 = compute_metrics(predictions_v2, ground_truth)`

In [ ]:
# YOUR CODE HERE
def classify_v2(response):
    pass  # same as v1 plus short-response rule

predictions_v2 = [classify_v2(r['response']) for r in outputs]
results_v2 = compute_metrics(predictions_v2, ground_truth)

print("Classifier v2 results:")
for k, v in results_v2.items():
    print(f"  {k}: {v}")

In [ ]:
check_equal(results_v2['tp'], 6, "v2 true positives")
check_approx(results_v2['recall'], 0.8571, 0.001, "v2 recall")
check_approx(results_v2['f1'], 0.8571, 0.001, "v2 f1")

## Step 3: Compare the Classifiers

Build a `comparison` dict with keys `'v1'` and `'v2'`, each holding its respective results dict.

Then set `recommended_classifier` to the string `'v1'` or `'v2'` — whichever has higher recall (since for safety filtering, recall is the priority metric).

In [ ]:
# YOUR CODE HERE
comparison = {}

recommended_classifier = ''  # 'v1' or 'v2'

# Print a summary table
print(f"{'Metric':<12} {'v1':>8} {'v2':>8}")
print("-" * 30)
for metric in ['precision', 'recall', 'f1', 'accuracy']:
    v1_val = results_v1.get(metric, 0)
    v2_val = results_v2.get(metric, 0)
    print(f"{metric:<12} {v1_val:>8.4f} {v2_val:>8.4f}")
print(f"\nRecommended for safety: {recommended_classifier}")

In [ ]:
check_type(comparison, dict, "comparison is a dict")
check_equal(recommended_classifier, 'v2', "recommended_classifier")

## Step 4: Write Your Findings

In a real research or engineering role, you would document your findings in a structured way — a memo, a report section, or a findings dict that feeds into a dashboard.

Build a `findings` dict with these keys:

- `'total_outputs'` — total number of outputs evaluated (int)
- `'total_flagged_ground_truth'` — number actually flagged in ground truth (int)
- `'v1_f1'` — F1 score for classifier v1 (float)
- `'v2_f1'` — F1 score for classifier v2 (float)
- `'recommended_classifier'` — `'v1'` or `'v2'` (string)
- `'key_weakness'` — a string describing what v1 misses (it misses `out_011` and `out_015`)
- `'key_finding'` — a string with your main takeaway about the two classifiers

Be specific in `'key_weakness'` and `'key_finding'` — write a real sentence, not a placeholder.

In [ ]:
# YOUR CODE HERE
findings = {
    'total_outputs': 0,
    'total_flagged_ground_truth': 0,
    'v1_f1': 0.0,
    'v2_f1': 0.0,
    'recommended_classifier': '',
    'key_weakness': '',
    'key_finding': ''
}

print("Findings:")
for k, v in findings.items():
    print(f"  {k}: {v!r}")

In [ ]:
check_keys(findings, ['total_outputs', 'total_flagged_ground_truth', 'v1_f1', 'v2_f1', 'recommended_classifier', 'key_weakness', 'key_finding'], "findings keys")
check_equal(findings['total_outputs'], 20, "total_outputs")
check_equal(findings['recommended_classifier'], 'v2', "recommended_classifier in findings")

## Reflection

You have completed the full Module 12 pipeline. Look back at what you built:

1. **Notebook 1** — Rule-based classifiers: trigger lists, confusion matrices
2. **Notebook 2** — Precision, recall, F1, and the asymmetric cost of safety errors
3. **Notebook 3** — Comparing two classifiers; understanding why accuracy alone is not enough
4. **Notebook 4** — End-to-end pipeline: implement, evaluate, compare, and document

### What you can take forward

- These same patterns apply to more complex classifiers — ML models, LLM judges, embedding-based classifiers.
- The `compute_metrics()` function you used here is a simplified version of what `sklearn.metrics` provides. In production you would likely use `classification_report()` from scikit-learn.
- The `findings` dict pattern translates directly to JSON reports, experiment tracking (e.g., MLflow, Weights & Biases), and research memos.
- The hardest decisions in safety engineering are not technical — they are about **how much recall is enough**, **what counts as harmful**, and **who decides**. Metrics give you a rigorous language for those conversations.

### Module 12 Complete